# Context-1 Colab Server (20b Optimized)

This notebook serves large models (up to 20b) from Colab, exposes an agent-style inference API, and is designed to sit behind Cloudflare.

### Optimization Notes:
- Uses `vllm` with memory optimizations for Colab T4/L4/A100.
- Supports streaming SSE for real-time agent output.
- 20b models on T4 (16GB) **require** a quantized version (AWQ/GPTQ) or will OOM.

In [ ]:
!nvidia-smi

## Configuration

In [ ]:
import os
from pathlib import Path

MODEL_NAME = os.getenv('MODEL_NAME', 'chromadb/context-1')
PUBLIC_HOSTNAME = os.getenv('PUBLIC_HOSTNAME', 'context1.pkking.computer')
VLLM_PORT = int(os.getenv('VLLM_PORT', '8001'))
API_PORT = int(os.getenv('API_PORT', '8000'))
WORKDIR = Path('/content/context1-server')
WORKDIR.mkdir(parents=True, exist_ok=True)
print({'MODEL_NAME': MODEL_NAME, 'PUBLIC_HOSTNAME': PUBLIC_HOSTNAME, 'VLLM_PORT': VLLM_PORT, 'API_PORT': API_PORT})

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb
!pip install -q --upgrade pip
!pip install -q vllm fastapi uvicorn requests sse-starlette huggingface_hub bitsandbytes accelerate

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = os.environ.get('HF_TOKEN') or userdata.get('HF_TOKEN')
if not hf_token:
    print('Warning: HF_TOKEN not found in secrets. Public models only.')
else:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)

In [ ]:
server_code = r'''
import json
import os
from typing import Any, Dict, List

import requests
from fastapi import FastAPI
from fastapi.responses import JSONResponse, StreamingResponse

VLLM_BASE_URL = f"http://127.0.0.1:{os.getenv('VLLM_PORT', '8001')}/v1"
MODEL_NAME = os.getenv('MODEL_NAME', 'chromadb/context-1')

app = FastAPI(title='Context-1 Agent Server')

def trajectory_to_messages(trajectory: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    messages = []
    for event in trajectory:
        role = event.get('role', 'user')
        content = event.get('content', '')
        if isinstance(content, (dict, list)):
            content = json.dumps(content)
        messages.append({'role': role, 'content': content})
    return messages or [{'role': 'user', 'content': 'Search for relevant supporting documents.'}]

def offline_payload() -> Dict[str, Any]:
    return {
        'error': {
            'type': 'service_unavailable',
            'message': 'The Colab-backed Context-1 runtime is not ready yet.'
        }
    }

def stream_vllm(payload: Dict[str, Any]):
    response = requests.post(f"{VLLM_BASE_URL}/chat/completions", json=payload, stream=True, timeout=600)
    response.raise_for_status()
    for line in response.iter_lines(decode_unicode=True):
        if not line:
            continue
        if line.startswith('data: '):
            yield line + '\n\n'
    yield 'data: [DONE]\n\n'

@app.get('/healthz')
def healthz():
    try:
        response = requests.get(f"{VLLM_BASE_URL}/models", timeout=15)
        response.raise_for_status()
        return {'status': 'ok', 'model': MODEL_NAME, 'provider': 'vllm'}
    except Exception as exc:
        return JSONResponse(status_code=503, content={'status': 'offline', 'error': str(exc)})

@app.post('/v1/agent/step')
def agent_step(payload: Dict[str, Any]):
    trajectory = payload.get('trajectory', [])
    tools = payload.get('tools', [])
    generation = payload.get('generation', {})
    request_payload = {
        'model': generation.get('model', MODEL_NAME),
        'messages': trajectory_to_messages(trajectory),
        'temperature': generation.get('temperature', 0),
        'max_tokens': generation.get('max_tokens', 1024),
        'stream': payload.get('stream', True),
    }
    if tools:
        request_payload['tools'] = tools
    if payload.get('stream', True):
        return StreamingResponse(stream_vllm(request_payload), media_type='text/event-stream')
    try:
        response = requests.post(f"{VLLM_BASE_URL}/chat/completions", json=request_payload, timeout=600)
        response.raise_for_status()
        return response.json()
    except Exception:
        return JSONResponse(status_code=503, content=offline_payload())
'''
from pathlib import Path
WORKDIR = Path(os.getenv('WORKDIR', '/content/context1-server'))
WORKDIR.mkdir(parents=True, exist_ok=True)
server_path = WORKDIR / 'context1_agent_server.py'
server_path.write_text(server_code)

In [ ]:
import os
import subprocess
from pathlib import Path

WORKDIR = Path(os.getenv('WORKDIR', '/content/context1-server'))
WORKDIR.mkdir(parents=True, exist_ok=True)
vllm_log = open(WORKDIR / 'vllm.log', 'w')
api_log = open(WORKDIR / 'api.log', 'w')

vllm_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_NAME,
    '--trust-remote-code',
    '--host', '127.0.0.1',
    '--port', str(VLLM_PORT),
    '--gpu-memory-utilization', '0.90',  # Maximize memory usage on Colab
    '--max-model-len', '4096',         # Cap context to prevent OOM
    '--enforce-eager'                  # Saves memory on smaller GPUs
]

# Add quantization flag if needed (e.g., if model name contains 'awq' or 'gptq')
if 'awq' in MODEL_NAME.lower():
    vllm_cmd.extend(['--quantization', 'awq'])
elif 'gptq' in MODEL_NAME.lower():
    vllm_cmd.extend(['--quantization', 'gptq'])

vllm_proc = subprocess.Popen(vllm_cmd, cwd=WORKDIR, stdout=vllm_log, stderr=subprocess.STDOUT)
api_proc = subprocess.Popen([
    'python', '-m', 'uvicorn', 'context1_agent_server:app',
    '--host', '0.0.0.0', '--port', str(API_PORT)
], cwd=WORKDIR, env={**os.environ, 'MODEL_NAME': MODEL_NAME, 'VLLM_PORT': str(VLLM_PORT)}, stdout=api_log, stderr=subprocess.STDOUT)
print({'vllm_pid': vllm_proc.pid, 'api_pid': api_proc.pid, 'vllm_log': str(WORKDIR / 'vllm.log'), 'api_log': str(WORKDIR / 'api.log')})

In [ ]:
import os
import time
from pathlib import Path
import requests

WORKDIR = Path(os.getenv('WORKDIR', '/content/context1-server'))
VLLM_HEALTH_URL = f'http://127.0.0.1:{VLLM_PORT}/v1/models'
API_HEALTH_URL = f'http://127.0.0.1:{API_PORT}/healthz'

for attempt in range(1, 61):
    try:
        vllm_response = requests.get(VLLM_HEALTH_URL, timeout=10)
        vllm_response.raise_for_status()
        api_response = requests.get(API_HEALTH_URL, timeout=10)
        api_response.raise_for_status()
        print(f'Attempt {attempt}: context1-ready')
        break
    except Exception as exc:
        print(f'Attempt {attempt}: waiting... ({str(exc)[:100]})')
        time.sleep(10)
else:
    for log_name in ['vllm.log', 'api.log']:
        log_path = WORKDIR / log_name
        if log_path.exists():
            print(f'\n===== {log_name} =====\n', log_path.read_text()[-4000:])
    raise RuntimeError('Context-1 services did not become ready within 10 minutes')

In [ ]:
import subprocess
from google.colab import userdata

cloudflared_token = os.environ.get('CLOUDFLARE_TUNNEL_TOKEN') or userdata.get('CLOUDFLARE_TUNNEL_TOKEN')
if not cloudflared_token:
    raise RuntimeError('Missing CLOUDFLARE_TUNNEL_TOKEN in Colab secrets')
os.environ['CLOUDFLARE_TUNNEL_TOKEN'] = cloudflared_token

cloudflared_proc = subprocess.Popen([
    'cloudflared', 'tunnel', 'run', '--token', cloudflared_token
])
print({'cloudflared_pid': cloudflared_proc.pid, 'public_hostname': PUBLIC_HOSTNAME})